# LlamaParse (2026년 최신 권장 사용법)

LlamaParse 는 LlamaIndex 에서 개발한 문서 파싱 서비스로, LLM 을 위해 특별히 설계되었습니다.

- PDF, Word, PowerPoint, Excel, HTML, 이미지 등 130개 이상의 형식 지원
- 자연어 지시(custom prompt)를 통한 맞춤형 파싱
- 복잡한 표·차트·이미지 추출
- 외국어(OCR 언어 지정) 지원

- 링크: https://cloud.llamaindex.ai

> **⚠️ 2026년 9월 기준 변경 사항**
>
> - 책에서 사용한 `llama-parse` / `llama-cloud-services` 패키지는 **deprecated**(2026-05-01 까지만 유지보수)입니다. 새 공식 SDK 는 **`llama-cloud`** (`from llama_cloud import LlamaCloud`) 이며, **Parse API v2** 를 사용합니다.
> - v2 에서는 옵션 이름 체계가 **티어(tier)** 중심으로 바뀌었습니다.
>
> | 책 (llama-parse, v1) | 현재 (llama-cloud, Parse v2) |
> |---|---|
> | `LlamaParse(result_type="markdown")` | `client.parsing.parse(..., expand=["markdown"])` (텍스트는 `"text"`, 전체 문자열은 `"markdown_full"`) |
> | `use_vendor_multimodal_model=True`, `vendor_multimodal_model_name="openai-gpt4o"`, `vendor_multimodal_api_key=...` | `tier="agentic"` 또는 `"agentic_plus"` (모델 선택·외부 API 키 불필요) |
> | `parsing_instruction="..."` | `agentic_options={"custom_prompt": "..."}` (`fast` 티어 제외) |
> | `language="ko"` | `processing_options={"ocr_parameters": {"languages": ["ko"]}}` |
> | `skip_diagonal_text=True` | `processing_options={"ignore": {"ignore_diagonal_text": True}}` |
> | `page_separator=...` | 페이지별 결과(`result.markdown.pages`)를 원하는 구분자로 직접 결합 |
> | `num_workers`, `nest_asyncio` | 불필요 (동기 클라이언트 / 비동기는 `AsyncLlamaCloud` + `await`) |
> | `SimpleDirectoryReader(file_extractor=...)` + `doc.to_langchain_format()` | 결과를 바로 LangChain `Document` 로 변환 (LlamaIndex 설치 불필요) |
>
> - 무료 사용량·크레딧 단가는 자주 바뀌므로 [요금 페이지](https://cloud.llamaindex.ai)에서 확인하세요.

**API 키 설정**
- API 키를 발급 후 `.env` 파일에 `LLAMA_CLOUD_API_KEY` 에 설정합니다.

In [ ]:
# 설치
# !pip install -qU llama-cloud langchain-core python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 기본 파서 적용

**티어(tier)** 선택 가이드
- `fast`: 가장 빠르고 저렴, 텍스트만 (Markdown 미지원)
- `cost_effective`: 텍스트 위주 문서에 적합한 균형형
- `agentic`: 이미지·도표가 있는 문서에 적합 (custom prompt 지원)
- `agentic_plus`: 복잡한 레이아웃·표에 최고 품질

In [ ]:
from pathlib import Path

from llama_cloud import LlamaCloud

FILE_PATH = "data/SPRI_AI_Brief_2023년12월호_F.pdf"

client = LlamaCloud()  # LLAMA_CLOUD_API_KEY 환경변수 사용

# 1) 파일 업로드
file_obj = client.files.create(file=Path(FILE_PATH), purpose="parse")

# 2) 파싱 (작업 제출 → 완료까지 대기 → 결과 조회를 SDK 가 처리)
result = client.parsing.parse(
    file_id=file_obj.id,
    tier="cost_effective",
    version="latest",
    processing_options={"ocr_parameters": {"languages": ["ko"]}},  # 구 language="ko"
    expand=["markdown"],  # 페이지별 Markdown 결과 포함
)

print(result.job.status)

In [ ]:
# 페이지 수 확인
len(result.markdown.pages)

In [ ]:
print(result.markdown.pages[0].markdown[:1000])

## LlamaParse 결과 → LangChain Document

책에서는 LlamaIndex `Document` 를 `to_langchain_format()` 으로 변환했지만, 이제는 결과 객체에서 바로 LangChain `Document` 를 만들면 됩니다.

In [ ]:
from langchain_core.documents import Document


def to_langchain_documents(result, source: str) -> list[Document]:
    docs = []
    for i, page in enumerate(result.markdown.pages, start=1):
        docs.append(
            Document(
                page_content=page.markdown,
                metadata={
                    "source": source,
                    "page": getattr(page, "page_number", i),  # v2 페이지 번호는 1부터
                    "job_id": result.job.id,
                    "parser": "llamaparse",
                },
            )
        )
    return docs


# 랭체인 도큐먼트로 변환
docs = to_langchain_documents(result, FILE_PATH)

In [ ]:
print(docs[5].page_content)

In [ ]:
# metadata 출력
docs[0].metadata

### 재사용을 위한 로더 클래스

다른 LangChain 로더와 같은 인터페이스(`load`, `lazy_load`)로 쓰고 싶다면 `BaseLoader` 로 감쌉니다.

In [ ]:
from typing import Any, Iterator

from langchain_core.document_loaders import BaseLoader


class LlamaParseLoader(BaseLoader):
    """llama-cloud Parse v2 를 LangChain 로더 인터페이스로 감싼 클래스"""

    def __init__(self, file_path: str, tier: str = "cost_effective", **parse_kwargs: Any) -> None:
        self.file_path = file_path
        self.tier = tier
        self.parse_kwargs = parse_kwargs
        self.client = LlamaCloud()

    def lazy_load(self) -> Iterator[Document]:
        file_obj = self.client.files.create(file=Path(self.file_path), purpose="parse")
        result = self.client.parsing.parse(
            file_id=file_obj.id,
            tier=self.tier,
            version="latest",
            expand=["markdown"],
            **self.parse_kwargs,
        )
        yield from to_langchain_documents(result, self.file_path)


loader = LlamaParseLoader(FILE_PATH, tier="cost_effective")
docs = loader.load()
len(docs)

## 고품질 파싱 (구 "MultiModal Model 로 파싱")

책에서는 `use_vendor_multimodal_model=True` 로 GPT-4o 를 붙여 파싱했습니다. Parse v2 에서는 **`agentic` / `agentic_plus` 티어**가 그 역할을 하므로 모델 이름이나 OpenAI API 키를 따로 넘길 필요가 없습니다.

**주요 파라미터**

- `tier`: `"agentic"` 또는 `"agentic_plus"` (AI 기반 파싱)
- `expand`: 결과로 받을 필드. `"markdown"`(페이지별), `"markdown_full"`(전체 1개 문자열), `"text"`, `"items"`(표·제목 등 구조화 트리) 등
- `processing_options.ocr_parameters.languages`: OCR 언어 (`["ko"]`)
- `processing_options.ignore.ignore_diagonal_text`: 대각선 텍스트(워터마크) 건너뛰기
- `page_ranges`: `{"target_pages": "1,3,5-10"}` 처럼 일부 페이지만 파싱 (1부터 시작)

In [ ]:
result = client.parsing.parse(
    file_id=file_obj.id,  # 앞에서 업로드한 파일 재사용
    tier="agentic",
    version="latest",
    processing_options={
        "ocr_parameters": {"languages": ["ko"]},
        # "ignore": {"ignore_diagonal_text": True},  # 구 skip_diagonal_text=True
    },
    expand=["markdown"],
)

# langchain 도큐먼트로 변환
docs = to_langchain_documents(result, FILE_PATH)

In [ ]:
print(docs[min(18, len(docs) - 1)].page_content)

페이지 구분자(구 `page_separator`)가 필요하면 페이지 결과를 직접 이어 붙입니다.

In [ ]:
PAGE_SEPARATOR = "\n=================\n"
full_markdown = PAGE_SEPARATOR.join(page.markdown for page in result.markdown.pages)
print(full_markdown[:1500])

## 사용자 정의 인스트럭션 (구 `parsing_instruction`)

`agentic_options.custom_prompt` 로 파싱 지시를 전달합니다. (`fast` 티어에서는 사용할 수 없습니다)

In [ ]:
# 파싱 지시문
custom_prompt = "You are parsing a brief of AI Report. Please extract tables in markdown format."

result = client.parsing.parse(
    file_id=file_obj.id,
    tier="agentic",
    version="latest",
    agentic_options={"custom_prompt": custom_prompt},
    processing_options={"ocr_parameters": {"languages": ["ko"]}},
    output_options={"markdown": {"tables": {"output_tables_as_markdown": True}}},
    expand=["markdown"],
)

# langchain 도큐먼트로 변환
docs = to_langchain_documents(result, FILE_PATH)

In [ ]:
# markdown 형식으로 추출된 테이블 확인
print(docs[-2].page_content)

## (추가) 구조화된 결과에서 표만 꺼내기

`expand` 에 `"items"` 를 추가하면 페이지별 요소 트리(제목·문단·표 등)를 받을 수 있습니다.

In [ ]:
result = client.parsing.parse(
    file_id=file_obj.id,
    tier="agentic",
    version="latest",
    expand=["markdown", "items"],
)

for page in result.items.pages:
    for item in page.items:
        if getattr(item, "type", None) == "table":
            print(f"page {page.page_number}: table")

## (추가) 비동기 사용

Jupyter 에서는 top-level `await` 로 바로 실행할 수 있습니다. (`nest_asyncio` 불필요)

In [ ]:
from llama_cloud import AsyncLlamaCloud

aclient = AsyncLlamaCloud()

afile = await aclient.files.create(file=Path(FILE_PATH), purpose="parse")
aresult = await aclient.parsing.parse(
    file_id=afile.id, tier="cost_effective", version="latest", expand=["markdown"]
)
len(aresult.markdown.pages)